# Scraper Performance Benchmark

This notebook presents an empirical performance evaluation of the arXiv data scraper developed for Milestone 1.
To ensure scientific reproducibility and fair comparison, all benchmarks are standardized on a **Google Colab CPU-only** environment.

### Notebook Workflow & Cache-First Architecture
1. **On Google Colab:** If benchmark data is not yet present, the notebook can execute a live micro-benchmark (50 representative papers: `2506.05788`–`2506.05837`) with periodic `psutil` memory sampling and export the raw telemetry.
2. **In CI/CD (GitHub Actions / Jupyter Book):** The notebook automatically detects pre-computed benchmark telemetry in `data/`, skipping redundant scraping and instantly generating all figures and tables in seconds.

> **Hardware Baseline:** Google Colab CPU-only runtime (Intel Xeon @ 2.20GHz, 2 vCPUs, ~12.7 GB RAM).

## 0. Environment Setup & Runtime Detection

In [1]:
import os
import sys
import json
import platform
import subprocess
import threading
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil

# ── Detect Google Colab Runtime ────────────────────────────────────────────────
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("[*] Running inside Google Colab.")
    # When running on Colab, ensure paths are aligned
    DATA_DIR = Path("data")
    DATA_DIR.mkdir(parents=True, exist_ok=True)
else:
    print("[*] Running in local / CI environment.")
    DATA_DIR = Path("data")

# ── Styling ────────────────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})
PRIMARY_BLUE = "#1f77b4"
ACCENT_RED   = "#d62728"
ACCENT_GREEN = "#2ca02c"
ACCENT_ORANGE= "#ff7f0e"

[*] Running in local / CI environment.


### 0.1 Cấu hình Bảo mật qua Google Colab Secrets (🔑)

Để chạy benchmark bảo mật và không bao giờ lộ token ra file nộp bài, bạn hãy thiết lập **2 Secrets** trong Google Colab (biểu tượng chiếc chìa khóa 🔑 ở thanh bên trái):

1. **`GITHUB_TOKEN`**: Personal Access Token (PAT) của GitHub có quyền truy cập repo `arXiv-Project`.
2. **`S2_API_KEY`**: API Key của Semantic Scholar (giúp tăng tốc lấy trích dẫn và giảm thời gian cooldown 429 từ 30s xuống 3s).

> *Lưu ý: Bật công tắc **Notebook access** sang màu xanh cho cả 2 keys để notebook tự động đọc.*

In [2]:
s2_api_key = None

if IN_COLAB:
    from pathlib import Path
    
    # 1. Cài đặt các gói phụ trợ cần thiết trên Colab
    print("[*] Đang cài đặt thư viện cần thiết (arxiv, psutil)...")
    !pip install arxiv psutil -q
    
    # 2. Đọc bảo mật từ Colab Secrets (🔑 ở thanh bên trái)
    print("[*] Đang đọc API Keys từ Colab Secrets...")
    try:
        from google.colab import userdata
        github_token = userdata.get('GITHUB_TOKEN')
        print("  -> GITHUB_TOKEN: Đã nhận diện thành công từ Secrets.")
    except Exception:
        import getpass
        github_token = getpass.getpass("Nhập GitHub Personal Access Token (PAT): ")
        
    try:
        from google.colab import userdata
        s2_api_key = userdata.get('S2_API_KEY')
        if s2_api_key:
            print("  -> S2_API_KEY: Đã nhận diện thành công từ Secrets.")
        else:
            print("  -> S2_API_KEY: Chưa thiết lập trong Secrets (sẽ chạy không dùng key).")
    except Exception:
        s2_api_key = None
        print("  -> S2_API_KEY: Không tìm thấy trong Secrets.")
    
    # 3. Đồng bộ mã nguồn crawler nhánh feat/benchmark-and-parallel-opt
    REPO_URL = f"https://{github_token}@github.com/QuangPhat101/arXiv-Project.git"
    BRANCH = "feat/benchmark-and-parallel-opt"
    PROJECT_DIR = Path("arXiv-Project")
    
    if not PROJECT_DIR.exists():
        print(f"[*] Đang clone arXiv-Project (nhánh: {BRANCH})...")
        !git clone -b {BRANCH} {REPO_URL} arXiv-Project
    else:
        print(f"[*] Thư mục arXiv-Project đã có. Đang đồng bộ nhánh {BRANCH}...")
        !git -C arXiv-Project fetch origin
        !git -C arXiv-Project checkout {BRANCH}
        !git -C arXiv-Project pull origin {BRANCH}
    
    print("[*] Toàn bộ môi trường và mã nguồn đã sẵn sàng!")

## 1. Testbed Specifications

The table below documents the standardized computing testbed. The specs are queried dynamically at runtime.

In [3]:
cpu_count = os.cpu_count() or 2
total_ram_gb = psutil.virtual_memory().total / (1024**3)

testbed_info = [
    ("Platform", "Google Colab" if IN_COLAB else f"Local ({platform.system()})"),
    ("Operating System", f"{platform.system()} {platform.release()}"),
    ("CPU Model", platform.processor() or "Intel Xeon (Cloud VM)"),
    ("Logical CPU Cores", f"{cpu_count} cores"),
    ("Total System RAM", f"{total_ram_gb:.2f} GB"),
    ("Python Runtime", platform.python_version()),
    ("arXiv Polite Delay", "3.0 seconds (rate-limit compliance)"),
    ("Semantic Scholar Delay", "1.5 seconds (free academic tier)"),
]

testbed_df = pd.DataFrame(testbed_info, columns=["Hardware / Setting", "Specification"]).set_index("Hardware / Setting")
display(testbed_df.style.set_caption("Table 1: Benchmark Testbed Specification"))

,Specification
Hardware / Setting,
Platform,Local (Linux)
Operating System,Linux 7.2.5-200.fc44.x86_64
CPU Model,Intel Xeon (Cloud VM)
Logical CPU Cores,8 cores
Total System RAM,19.22 GB
Python Runtime,3.14.7
arXiv Polite Delay,3.0 seconds (rate-limit compliance)
Semantic Scholar Delay,1.5 seconds (free academic tier)


## 2. Baseline Experiment: Sequential Crawler

The original baseline crawler operates in a **synchronous sequential pipeline**:
1. **Phase 1 (Discovery):** Queries arXiv API in batches to resolve version history and metadata.
2. **Phase 2 (Acquisition & Clean-up):** For each paper in sequence:
   - Downloads raw tarball from arXiv (with 3.0s polite delay).
   - Selectively extracts `.tex` and `.bib` files, discarding all non-text assets (PDFs, high-resolution figures).
   - Fetches reference citations via Semantic Scholar API.

### Micro-Benchmark Setup (50 Papers)
To accurately profile RAM and CPU consumption without risking Colab disconnection timeouts,
we benchmark a 50-paper window: `2506.05788` to `2506.05837`.

In [4]:
def print_benchmark_summary(data: dict) -> None:
    run = data.get("run_summary", {})
    mem = data.get("memory", {})
    stor = data.get("storage", {})
    phases = data.get("phase_breakdown_s", {})

    print("=" * 65)
    print("          BENCHMARK PROFILER SUMMARY REPORT          ")
    print("=" * 65)
    print(f" Evaluated Papers:        {run.get('evaluated_papers', 0)}")
    print(f" Total Wall-clock Time:   {run.get('wall_time_s', 0.0):.1f} s ({run.get('wall_time_min', 0.0):.2f} mins)")
    print(f" Average Time per Paper:  {run.get('avg_time_per_paper_s', 0.0):.2f} s")
    print(f" Peak Memory (RSS):       {mem.get('peak_ram_mb', 0.0):.1f} MB")
    print(f" Average Memory (RSS):    {mem.get('avg_ram_mb', 0.0):.1f} MB")
    print(f" Total Raw Download:      {stor.get('total_raw_mb', 0.0):.2f} MB")
    print(f" Total Extracted (.tex):  {stor.get('total_extracted_mb', 0.0):.2f} MB")
    print(f" Storage Savings:         {stor.get('storage_savings_pct', 0.0):.1f} %")
    print("-" * 65)
    print(" Phase Duration Breakdown:")
    total_phase = sum(phases.values()) or 1.0
    for phase, dur in phases.items():
        pct = (dur / total_phase) * 100
        print(f"   - {phase:<18}: {dur:>6.1f} s  ({pct:>5.1f}%)")
    print("=" * 65)

# Locate or prepare baseline telemetry file
candidates = [
    DATA_DIR / "baseline_colab_stats.json",
    Path("data/baseline_colab_stats.json"),
    Path("milestone1/data/baseline_colab_stats.json"),
]
BASELINE_STATS_PATH = candidates[0]
for p in candidates:
    if p.exists():
        BASELINE_STATS_PATH = p
        break

baseline_data = None

# ── Cache-First Execution Logic ────────────────────────────────────────────────
if not BASELINE_STATS_PATH.exists():
    crawler_script = Path("src/crawler.py")
    crawler_cwd = Path(".")
    if not crawler_script.exists() and IN_COLAB:
        if Path("arXiv-Project/src/crawler.py").exists():
            crawler_script = Path("arXiv-Project/src/crawler.py")
            crawler_cwd = Path("arXiv-Project")

    if crawler_script.exists():
        print("[*] Running live baseline benchmark (50 papers) on Colab...")
        print("[*] Streaming per-paper telemetry live below:
")

        OUTPUT_TEMP = Path("./data_colab_baseline_temp").resolve()
        OUTPUT_TEMP.mkdir(parents=True, exist_ok=True)

        cmd = [
            sys.executable, "-u", str(crawler_script.resolve()),
            "--month", "2506",
            "--start", "5788",
            "--end", "5837",
            "--output", str(OUTPUT_TEMP),
            "--profile",
            "--profile-output", str(BASELINE_STATS_PATH.resolve()),
        ]
        if "s2_api_key" in globals() and s2_api_key:
            cmd.extend(["--s2-api-key", str(s2_api_key)])

        start_wall = time.perf_counter()
        proc = subprocess.Popen(
            cmd,
            cwd=str(crawler_cwd),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )

        # Real-time streaming output to Colab cell during crawling
        for line in proc.stdout:
            print(line, end="", flush=True)

        proc.wait()
        wall_time = time.perf_counter() - start_wall

        # Clear verbose streaming logs so the cell retains ONLY the clean summary report
        try:
            from IPython.display import clear_output
            clear_output(wait=True)
        except Exception:
            pass

        if BASELINE_STATS_PATH.exists():
            with open(BASELINE_STATS_PATH, encoding="utf-8") as f:
                baseline_data = json.load(f)
            print(f"[*] Benchmark completed in {wall_time:.1f}s. Loaded from '{BASELINE_STATS_PATH}'
")
            print_benchmark_summary(baseline_data)
        else:
            print(f"
[!] ERROR: Benchmark finished (exit code {proc.returncode}), but '{BASELINE_STATS_PATH}' was not found!")
    else:
        print(f"[*] '{BASELINE_STATS_PATH}' not found.")
        print("[*] NOTE: Crawler code not found locally. To run the benchmark live, clone or upload arXiv-Project to Colab.")
        print("[*] Alternatively, ensure 'data/baseline_colab_stats.json' exists to view offline report.")
else:
    with open(BASELINE_STATS_PATH, encoding="utf-8") as f:
        baseline_data = json.load(f)
    print(f"[*] Loaded existing benchmark telemetry: '{BASELINE_STATS_PATH}'
")
    print_benchmark_summary(baseline_data)


In [5]:
# Fallback check if baseline_data wasn't loaded
if baseline_data is None:
    for p in [DATA_DIR / "baseline_colab_stats.json", Path("data/baseline_colab_stats.json"), Path("milestone1/data/baseline_colab_stats.json")]:
        if p.exists():
            with open(p, encoding="utf-8") as f:
                baseline_data = json.load(f)
            break

if baseline_data is not None:
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 4.8))
    
    mem_info = baseline_data.get("memory", {})
    run_info = baseline_data.get("run_summary", {})
    stor_info = baseline_data.get("storage", {})
    phases = baseline_data.get("phase_breakdown_s", {})
    
    # ── Figure 1: RAM Timeline ─────────────────────────────────────────────────
    t = mem_info.get("timeline_s", [])
    ram = mem_info.get("ram_timeline_mb", [])
    peak_ram = mem_info.get("peak_ram_mb", 0.0)
    avg_ram = mem_info.get("avg_ram_mb", 0.0)
    
    if t and ram:
        ax1.plot(t, ram, color=PRIMARY_BLUE, linewidth=1.5, label="RSS Memory (MB)")
        ax1.axhline(peak_ram, color=ACCENT_RED, linestyle="--", label=f"Peak = {peak_ram:.1f} MB")
        ax1.axhline(avg_ram, color=ACCENT_ORANGE, linestyle=":", label=f"Avg = {avg_ram:.1f} MB")
        ax1.fill_between(t, ram, alpha=0.15, color=PRIMARY_BLUE)
        ax1.set_xlabel("Elapsed Time (s)")
        ax1.set_ylabel("Resident Memory (MB)")
        ax1.set_title("Figure 1: Baseline Memory Footprint (Google Colab CPU)")
        ax1.legend(loc="lower right")
    
    # ── Figure 2: CPU Utilization ──────────────────────────────────────────────
    cpu = mem_info.get("cpu_timeline_pct", [])
    if t and cpu:
        ax2.plot(t, cpu, color=ACCENT_GREEN, linewidth=1.2, label="CPU Utilization (%)")
        ax2.set_xlabel("Elapsed Time (s)")
        ax2.set_ylabel("CPU (%)")
        ax2.set_title("Figure 2: CPU Utilization (Idle intervals during rate-limit sleep)")
        ax2.set_ylim(0, 105)
        ax2.legend(loc="upper right")
        
    # ── Figure 3: Phase Time Breakdown (Bottlenecks) ───────────────────────────
    if phases:
        phase_names = [
            ("download", "Download"),
            ("s2_references", "S2 References"),
            ("rate_limit_sleep", "Rate-limit Sleep"),
            ("extract", "Extract (.tex)"),
            ("discovery", "Discovery"),
        ]
        ordered_keys = [k for k, _ in phase_names if k in phases]
        labels = [lbl for k, lbl in phase_names if k in phases]
        durations = [phases[k] for k in ordered_keys]
        
        y_pos = np.arange(len(durations))
        bar_colors = [PRIMARY_BLUE, ACCENT_ORANGE, "#9467bd", ACCENT_GREEN, "#8c564b"][:len(durations)]
        bars = ax3.barh(y_pos, durations, color=bar_colors, height=0.55)
        ax3.set_yticks(y_pos)
        ax3.set_yticklabels(labels)
        ax3.set_xlabel("Execution Time (s)")
        ax3.set_title("Figure 3: Cumulative Phase Durations (Bottlenecks)")
        
        total_p = sum(durations) or 1.0
        for i, (bar, dur) in enumerate(zip(bars, durations)):
            pct = (dur / total_p) * 100
            ax3.text(dur + max(total_p * 0.01, 1.0), bar.get_y() + bar.get_height() / 2,
                     f"{dur:.1f}s ({pct:.1f}%)", va="center", fontsize=9)
        ax3.set_xlim(0, max(durations) * 1.25 if durations else 10)
    
    plt.tight_layout()
    plt.show()
    
    # ── Summary Table ──────────────────────────────────────────────────────────
    summary_rows = [
        ("Evaluated Papers", f"{run_info.get('evaluated_papers', 50)} papers"),
        ("Total Wall-clock Time", f"{run_info.get('wall_time_s', 0.0):.1f} s ({run_info.get('wall_time_min', 0.0):.2f} mins)"),
        ("Average Time per Paper", f"{run_info.get('avg_time_per_paper_s', 0.0):.2f} s"),
        ("Peak Memory (RSS)", f"{peak_ram:.1f} MB"),
        ("Average Memory (RSS)", f"{avg_ram:.1f} MB"),
        ("Total Raw Download", f"{stor_info.get('total_raw_mb', 0.0):.2f} MB"),
        ("Total Extracted (.tex/.bib)", f"{stor_info.get('total_extracted_mb', 0.0):.2f} MB"),
        ("Storage Savings", f"{stor_info.get('storage_savings_pct', 0.0):.1f}%"),
    ]
    summary_df = pd.DataFrame(summary_rows, columns=["Metric", "Baseline Measurement"]).set_index("Metric")
    display(summary_df.style.set_caption("Table 2: Baseline Performance Summary"))
    
    # ── Phase Breakdown Table ──────────────────────────────────────────────────
    if phases:
        tot_time = sum(phases.values()) or 1.0
        breakdown_rows = [
            (phase, f"{dur:.2f} s", f"{(dur / tot_time) * 100:.1f}%")
            for phase, dur in phases.items()
        ]
        breakdown_df = pd.DataFrame(breakdown_rows, columns=["Pipeline Phase", "Cumulative Duration", "Proportion"]).set_index("Pipeline Phase")
        display(breakdown_df.style.set_caption("Table 3: Micro-Phase Duration Breakdown"))
else:
    print("[*] Ready for benchmark execution. Run Cell 8 above to produce telemetry data.")


## 3. Optimization: Producer-Consumer Parallel Extraction

*(This section will document the parallelized pipeline implementation, A/B comparative benchmarks, and Amdahl's Law analysis in the next stage).* 